In [ ]:
import os
import re
import json
import math
import time
import random
import datetime
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF

import segmentation_models_pytorch as smp

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

In [ ]:
CONFIG = {
    # ---- paths --------------------------------------------------------
    'images_root'     : ['./sd2-fr',     './sdxl-fr'    ],
    'masks_root'      : ['./masks-sd2',  './masks-sdxl' ],
    'save_dir'        : './checkpoints_unet',   # separate from DeepLab checkpoints

    # ---- model --------------------------------------------------------
    # UNet-specific notes:
    #   encoder: ResNet-50 keeps parity with DeepLab for fair comparison.
    #            Swap to 'resnet101' or 'efficientnet-b4' for more capacity.
    #   decoder_channels: feature channels at each decoder stage.
    #     (256,128,64,32,16) is the smp default — good starting point.
    #   decoder_dropout: applied inside each decoder block.
    #     UNet decoders are shallower than DeepLab's ASPP head, so dropout
    #     here has a stronger regularisation effect per layer.
    'encoder'             : 'resnet50',
    'encoder_weights'     : 'imagenet',
    'num_classes'         : 1,
    'decoder_channels'    : (256, 128, 64, 32, 16),
    'decoder_dropout'     : 0.3,

    # ---- training -----------------------------------------------------
    'image_size'          : 512,
    'batch_size'          : 32,
    'epochs'              : 30,
    'lr'                  : 1e-4,
    'weight_decay'        : 5e-3,
    'num_workers'         : 16,
    'persistent_workers'  : True,
    'prefetch_factor'     : 4,
    'use_amp'             : True,
    'seed'                : 42,

    # ---- gradient clipping --------------------------------------------
    'grad_clip'           : 1.0,

    # ---- early stopping -----------------------------------------------
    'early_stop_patience' : 5,

    # ---- loss ---------------------------------------------------------
    'pos_weight'          : 3.0,
    'loss_alpha'          : 0.4,   # BCE weight
    'loss_beta'           : 0.2,   # Focal weight  (remainder 0.4 -> Dice)
    'focal_gamma'         : 2.0,
    'label_smoothing'     : 0.05,

    # ---- augmentation -------------------------------------------------
    'aug_strength'        : 'heavy',

    # ---- CutMix -------------------------------------------------------
    'cutmix_prob'         : 0.3,
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device          : {DEVICE}")

In [ ]:
def extract_id(filename: str):
    m = re.match(r'^(\d+)_', filename)
    return m.group(1) if m else None


def infer_mask_suffix(masks_root: str) -> str:
    for p in Path(masks_root).rglob('*.png'):
        m = re.match(r'^\d+_mask_(\d+)\.png$', p.name)
        if m:
            return f"_mask_{m.group(1)}.png"
    return '_mask_512.png'


def build_pairs(split: str, images_root: str, masks_root: str):
    img_root    = Path(images_root) / split
    mask_root   = Path(masks_root)  / split
    mask_suffix = infer_mask_suffix(masks_root)

    if not img_root.exists():
        raise FileNotFoundError(f"Images split folder not found: {img_root}")
    if not mask_root.exists():
        raise FileNotFoundError(f"Masks split folder not found:  {mask_root}")

    pairs, missing = [], []

    for category_dir in sorted(img_root.iterdir()):
        if not category_dir.is_dir():
            continue
        mask_cat_dir = mask_root / category_dir.name

        for img_path in sorted(category_dir.glob('*.png')):
            img_id = extract_id(img_path.name)
            if img_id is None:
                continue
            mask_path = mask_cat_dir / f"{img_id}{mask_suffix}"
            if mask_path.exists():
                pairs.append((img_path, mask_path))
            else:
                missing.append(str(mask_path))

    if missing:
        print(f"  [{split}|{Path(images_root).name}] WARNING: {len(missing)} masks not found "
              f"(suffix: {mask_suffix}, first 5):")
        for p in missing[:5]:
            print(f"    {p}")

    print(f"  [{split}|{Path(images_root).name}] {len(pairs):,} pairs  "
          f"(mask suffix: {mask_suffix})")
    return pairs


def build_all_pairs(split: str, config: dict) -> list:
    img_roots  = config['images_root']
    mask_roots = config['masks_root']
    if isinstance(img_roots,  str): img_roots  = [img_roots]
    if isinstance(mask_roots, str): mask_roots = [mask_roots]

    if len(img_roots) != len(mask_roots):
        raise ValueError(f"images_root ({len(img_roots)}) and masks_root "
                         f"({len(mask_roots)}) must have the same length.")

    all_pairs = []
    for img_r, mask_r in zip(img_roots, mask_roots):
        all_pairs.extend(build_pairs(split, img_r, mask_r))

    print(f"  [{split}] TOTAL: {len(all_pairs):,} pairs across {len(img_roots)} dataset(s)\n")
    return all_pairs


print("\nVerifying dataset paths...")
for _split in ('training', 'validation', 'testing'):
    _pairs = build_all_pairs(_split, CONFIG)
    if _pairs:
        _ip, _mp = _pairs[0]
        print(f"    sample  image : {_ip.name}")
        print(f"    sample  mask  : {_mp.name}")
print()

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


class InpaintingSegDataset(Dataset):
    """
    Binary segmentation dataset.
    image : float32 [3, H, W]  normalised to ImageNet stats
    mask  : float32 [1, H, W]  values in {0.0, 1.0}
    """

    def __init__(self, pairs: list, image_size: int, mode: str = 'train'):
        self.pairs      = pairs
        self.image_size = image_size
        self.mode       = mode

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        img  = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        img  = img.resize( (self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        if self.mode == 'train':
            strength = CONFIG.get('aug_strength', 'medium')

            # geometric (identical on image AND mask)
            if torch.rand(1).item() > 0.5:
                img  = TF.hflip(img);  mask = TF.hflip(mask)
            if torch.rand(1).item() > 0.5:
                img  = TF.vflip(img);  mask = TF.vflip(mask)

            if strength in ('medium', 'heavy'):
                k = random.randint(0, 3)
                if k > 0:
                    img  = TF.rotate(img,  90 * k)
                    mask = TF.rotate(mask, 90 * k)

            if strength == 'heavy':
                scale     = 0.7 + torch.rand(1).item() * 0.3
                crop_size = int(self.image_size * scale)
                max_off   = self.image_size - crop_size
                top  = random.randint(0, max_off)
                left = random.randint(0, max_off)
                img  = TF.resized_crop(img,  top, left, crop_size, crop_size,
                                       (self.image_size, self.image_size),
                                       interpolation=Image.BILINEAR)
                mask = TF.resized_crop(mask, top, left, crop_size, crop_size,
                                       (self.image_size, self.image_size),
                                       interpolation=Image.NEAREST)

            # photometric (image only)
            if strength in ('medium', 'heavy'):
                img = TF.adjust_brightness(img, 1.0 + (torch.rand(1).item() - 0.5) * 0.4)
                img = TF.adjust_contrast  (img, 1.0 + (torch.rand(1).item() - 0.5) * 0.4)
                img = TF.adjust_saturation(img, 1.0 + (torch.rand(1).item() - 0.5) * 0.2)
                img = TF.adjust_hue       (img, (torch.rand(1).item() - 0.5) * 0.1)

            if strength == 'heavy':
                if torch.rand(1).item() < 0.15:
                    img = TF.to_grayscale(img, num_output_channels=3)
                if torch.rand(1).item() < 0.20:
                    import PIL.ImageFilter
                    img = img.filter(PIL.ImageFilter.GaussianBlur(
                        radius=random.choice([1, 2, 3])))

        img  = TF.to_tensor(img)
        img  = TF.normalize(img, mean=MEAN, std=STD)

        mask_np = np.array(mask, dtype=np.float32) / 255.0
        mask_t  = torch.from_numpy(mask_np).unsqueeze(0)

        return img, mask_t


def make_loaders(config):
    train_pairs = build_all_pairs('training',   config)
    val_pairs   = build_all_pairs('validation', config)
    test_pairs  = build_all_pairs('testing',    config)

    train_ds = InpaintingSegDataset(train_pairs, config['image_size'], mode='train')
    val_ds   = InpaintingSegDataset(val_pairs,   config['image_size'], mode='val')
    test_ds  = InpaintingSegDataset(test_pairs,  config['image_size'], mode='test')

    kw = dict(
        batch_size         = config['batch_size'],
        num_workers        = config['num_workers'],
        pin_memory         = True,
        persistent_workers = config.get('persistent_workers', True),
        prefetch_factor    = config.get('prefetch_factor', 4),
    )

    train_loader = DataLoader(train_ds, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)

    print(f"\nDataset sizes  --  "
          f"Train: {len(train_ds):,}  |  "
          f"Val: {len(val_ds):,}  |  "
          f"Test: {len(test_ds):,}")
    print(f"Steps per epoch: {len(train_loader):,}  "
          f"(batch_size={config['batch_size']})")
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(CONFIG)

In [ ]:
def unnormalize(t):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(MEAN).view(3,1,1)
    std  = torch.tensor(STD ).view(3,1,1)
    return (t * std + mean).clamp(0, 1)


imgs, masks = next(iter(train_loader))
n_show = min(4, len(imgs))

fig, axes = plt.subplots(n_show, 2, figsize=(8, n_show * 4))
for i in range(n_show):
    img_disp  = unnormalize(imgs[i]).permute(1, 2, 0).numpy()
    mask_disp = masks[i].squeeze().numpy()

    axes[i, 0].imshow(img_disp)
    axes[i, 0].set_title('SD2 Image', fontsize=11)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask_disp, cmap='gray', vmin=0, vmax=1)
    axes[i, 1].set_title('Mask (white = AI inpainted)', fontsize=11)
    axes[i, 1].axis('off')

plt.suptitle('Sample batch — verify image↔mask alignment', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"Image batch shape: {imgs.shape}   Mask batch shape: {masks.shape}")
print(f"Mask unique values: {masks.unique().tolist()}")

In [ ]:
# =============================================================================
# SECTION 5: MODEL  — UNet with ImageNet-pretrained ResNet-50 encoder
# =============================================================================
#
# Key architectural differences from DeepLabV3+:
#
#   DeepLabV3+                          UNet
#   ─────────────────────────────────   ──────────────────────────────────
#   ASPP captures multi-scale context   Skip connections preserve spatial detail
#   Strong on global structure          Strong on boundaries & fine edges
#   Single-scale decoder                Multi-scale decoder (one per encoder stage)
#   Good for large regions              Good for precise masks
#
# For inpainting detection, both are valid. UNet often wins on thin/small
# inpainted regions because skip connections carry high-res encoder features
# all the way to the output. DeepLab wins when the inpainted region is large
# and context-dependent.
#
# Dropout placement:
#   In DeepLabV3+ we injected Dropout2d after decoder.block2.
#   In smp's UNet the decoder is a sequence of DecoderBlock modules.
#   Each DecoderBlock already has a Conv-BN-ReLU stack.
#   We add Dropout2d after the last conv in each decoder block,
#   giving regularisation at every resolution level of the decoder.

def build_model(config):
    model = smp.Unet(
        encoder_name    = config['encoder'],           # 'resnet50'
        encoder_weights = config['encoder_weights'],   # 'imagenet'
        in_channels     = 3,
        classes         = config['num_classes'],       # 1 for binary
        activation      = None,                        # raw logits
        decoder_channels= config.get('decoder_channels', (256, 128, 64, 32, 16)),
        # decoder_use_batchnorm: True is the smp default and strongly recommended.
        # BN stabilises training, especially with the aggressive augmentation we use.
        decoder_use_batchnorm = True,
    )

    # Inject Dropout2d after the conv block inside each UNet decoder stage.
    # smp's UNet decoder is: model.decoder.blocks  (a ModuleList of DecoderBlock)
    # Each DecoderBlock has a .conv attribute which is a Sequential of two
    # Conv2d-BN-ReLU stacks. We append Dropout2d after the full conv block.
    dropout_p = config.get('decoder_dropout', 0.3)
    if dropout_p > 0:
        for i, block in enumerate(model.decoder.blocks):
            # Linear dropout schedule: early (high-res) decoder stages get less
            # dropout because they handle fine spatial detail that is easy to
            # destroy; later stages handle semantic features and can afford more.
            # Stage 0 (highest res) -> p * 0.4
            # Stage 4 (lowest  res) -> p * 1.0
            n      = len(model.decoder.blocks)
            scaled = dropout_p * (0.4 + 0.6 * i / max(n - 1, 1))
            block.conv = nn.Sequential(
                block.conv,
                nn.Dropout2d(p=scaled),
            )
        print(f"  Dropout2d injected into {n} decoder blocks  "
              f"(p range: {dropout_p*0.4:.2f} - {dropout_p:.2f})")

    return model

In [ ]:
class CombinedLoss(nn.Module):
    """
    Loss = alpha * BCE(pos_weight, smoothed)
         + beta  * Focal(smoothed)
         + (1 - alpha - beta) * Dice(per-image, hard targets)

    Label smoothing is applied to BCE and Focal only.
    Dice stays on hard {0,1} targets so the overlap metric remains clean.
    """

    def __init__(self, pos_weight: float = 3.0,
                 alpha: float = 0.4,
                 beta:  float = 0.2,
                 focal_gamma: float = 2.0,
                 label_smoothing: float = 0.05):
        super().__init__()
        self.bce            = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
        self.alpha          = alpha
        self.beta           = beta
        self.gamma          = focal_gamma
        self.dice_weight    = 1.0 - alpha - beta
        self.label_smoothing = label_smoothing

    def _smooth(self, targets):
        eps = self.label_smoothing
        return targets * (1.0 - eps) + (1.0 - targets) * eps

    def _dice(self, logits, targets, smooth=1.0):
        B     = logits.shape[0]
        probs = torch.sigmoid(logits).view(B, -1)
        tgts  = targets.view(B, -1)
        inter = (probs * tgts).sum(dim=1)
        union = probs.sum(dim=1) + tgts.sum(dim=1)
        return (1.0 - (2.0 * inter + smooth) / (union + smooth)).mean()

    def _focal(self, logits, targets):
        if self.gamma == 0 or self.beta == 0:
            return torch.tensor(0.0, device=logits.device)
        probs        = torch.sigmoid(logits)
        p_t          = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1.0 - p_t) ** self.gamma
        bce_elem     = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        return (focal_weight * bce_elem).mean()

    def forward(self, logits, targets):
        smooth_t  = self._smooth(targets) if self.label_smoothing > 0 else targets
        bce_val   = self.bce(logits, smooth_t)
        focal_val = self._focal(logits, smooth_t)
        dice_val  = self._dice(logits, targets)
        return (self.alpha       * bce_val
              + self.beta        * focal_val
              + self.dice_weight * dice_val)


# =============================================================================
# SECTION 7: METRICS
# =============================================================================

def compute_metrics(logits: torch.Tensor,
                    targets: torch.Tensor,
                    threshold: float = 0.5) -> dict:
    preds   = (torch.sigmoid(logits) > threshold).float().view(-1)
    targets = targets.view(-1)

    TP = (preds * targets).sum().item()
    FP = (preds * (1 - targets)).sum().item()
    FN = ((1 - preds) * targets).sum().item()

    eps = 1e-6
    iou       = TP / (TP + FP + FN + eps)
    dice      = (2 * TP) / (2 * TP + FP + FN + eps)
    precision = TP / (TP + FP + eps)
    recall    = TP / (TP + FN + eps)
    f1        = (2 * precision * recall) / (precision + recall + eps)

    return dict(iou=iou, dice=dice, precision=precision, recall=recall, f1=f1)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, epoch, total_epochs):
    model.train()
    total_loss  = 0.0
    sum_metrics = dict(iou=0.0, dice=0.0, precision=0.0, recall=0.0, f1=0.0)
    cutmix_prob = CONFIG.get('cutmix_prob', 0.0)

    bar = tqdm(
        loader,
        desc=f"Epoch {epoch:02d}/{total_epochs} [train]",
        unit='batch',
        dynamic_ncols=True,
        colour='green',
    )

    for step, (images, masks) in enumerate(bar, start=1):
        images, masks = images.to(device), masks.to(device)

        # CutMix: paste a random crop from one image onto another.
        # Prevents the model memorising object shapes as proxies for fakeness.
        if cutmix_prob > 0 and torch.rand(1).item() < cutmix_prob:
            B, C, H, W = images.shape
            lam      = float(torch.distributions.Beta(1.0, 1.0).sample())
            cut_h    = int(H * (1.0 - lam) ** 0.5)
            cut_w    = int(W * (1.0 - lam) ** 0.5)
            cx       = torch.randint(W, (1,)).item()
            cy       = torch.randint(H, (1,)).item()
            x1 = max(cx - cut_w // 2, 0); x2 = min(cx + cut_w // 2, W)
            y1 = max(cy - cut_h // 2, 0); y2 = min(cy + cut_h // 2, H)
            perm = torch.randperm(B, device=device)
            images[:, :, y1:y2, x1:x2] = images[perm, :, y1:y2, x1:x2]
            masks[ :, :, y1:y2, x1:x2] = masks[ perm, :, y1:y2, x1:x2]

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            logits = model(images)
            loss   = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if CONFIG.get('grad_clip', 0) > 0:
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        m = compute_metrics(logits.detach(), masks.detach())
        for k in sum_metrics:
            sum_metrics[k] += m[k]

        bar.set_postfix(
            loss =f"{total_loss / step:.4f}",
            iou  =f"{sum_metrics['iou']  / step:.3f}",
            dice =f"{sum_metrics['dice'] / step:.3f}",
        )

    n = len(loader)
    return total_loss / n, {k: v / n for k, v in sum_metrics.items()}


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc='val'):
    model.eval()
    total_loss  = 0.0
    sum_metrics = dict(iou=0.0, dice=0.0, precision=0.0, recall=0.0, f1=0.0)

    bar = tqdm(
        loader,
        desc=f"              [{desc}] ",
        unit='batch',
        dynamic_ncols=True,
        colour='cyan',
        leave=False,
    )

    for step, (images, masks) in enumerate(bar, start=1):
        images, masks = images.to(device), masks.to(device)

        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            logits = model(images)
            loss   = criterion(logits, masks)

        total_loss += loss.item()
        m = compute_metrics(logits, masks)
        for k in sum_metrics:
            sum_metrics[k] += m[k]

        bar.set_postfix(
            loss=f"{total_loss / step:.4f}",
            iou =f"{sum_metrics['iou'] / step:.3f}",
        )

    n = len(loader)
    return total_loss / n, {k: v / n for k, v in sum_metrics.items()}


In [ ]:
os.makedirs(CONFIG['save_dir'], exist_ok=True)

# --- data ---
print("=" * 65)
print("Setting up data loaders...")
print("=" * 65)
train_loader, val_loader, test_loader = make_loaders(CONFIG)

# --- model ---
print("\nBuilding UNet model...")
model = build_model(CONFIG).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params     : {total_p:,}")
print(f"  Trainable params : {trainable_p:,}")

with torch.no_grad():
    _d = torch.randn(2, 3, CONFIG['image_size'], CONFIG['image_size']).to(DEVICE)
    _o = model(_d)
    print(f"  Forward test     : input {list(_d.shape)} -> output {list(_o.shape)}  OK")

# --- loss ---
criterion = CombinedLoss(
    pos_weight      = CONFIG.get('pos_weight',      3.0),
    alpha           = CONFIG.get('loss_alpha',      0.4),
    beta            = CONFIG.get('loss_beta',       0.2),
    focal_gamma     = CONFIG.get('focal_gamma',     2.0),
    label_smoothing = CONFIG.get('label_smoothing', 0.05),
).to(DEVICE)

# --- optimiser ---
# UNet has more decoder parameters than DeepLab (5 decoder stages vs 1 head).
# We apply differential LR the same way:
#   encoder (pretrained ResNet) : lr * 0.1  -- don't disturb ImageNet features
#   decoder + head (random init): lr        -- learn freely
optimizer = optim.AdamW(
    [
        {'params': model.encoder.parameters(),           'lr': CONFIG['lr'] * 0.1},
        {'params': model.decoder.parameters(),           'lr': CONFIG['lr']},
        {'params': model.segmentation_head.parameters(), 'lr': CONFIG['lr']},
    ],
    weight_decay = CONFIG['weight_decay'],
)

# Warmup (3 epochs linear ramp) + cosine decay.
# Warmup prevents large early gradients from destroying pretrained encoder
# features before the randomly-initialised decoder has stabilised.
warmup_epochs = 3
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(CONFIG['epochs'] - warmup_epochs, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=CONFIG['use_amp'])

print(f"\n  Encoder LR : {CONFIG['lr'] * 0.1:.1e}")
print(f"  Decoder LR : {CONFIG['lr']:.1e}")

# --- training loop ---
history = {k: [] for k in
            ('train_loss', 'val_loss', 'train_iou', 'val_iou',
            'train_dice', 'val_dice')}

best_val_iou      = 0.0
best_epoch        = 0
best_metrics      = {}
epochs_no_improve = 0
patience          = CONFIG.get('early_stop_patience', 5)

print("\n" + "=" * 65)
print(f"Starting UNet training  --  {CONFIG['epochs']} epochs  |  device: {DEVICE}")
print(f"Early stopping patience : {patience} epochs")
print("=" * 65 + "\n")

training_start = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    epoch_start = time.time()

    train_loss, train_m = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler,
        DEVICE, epoch, CONFIG['epochs']
    )
    val_loss, val_m = evaluate(model, val_loader, criterion, DEVICE, desc='val')
    scheduler.step()

    epoch_secs = time.time() - epoch_start

    history['train_loss'].append(train_loss)
    history['val_loss'  ].append(val_loss)
    history['train_iou' ].append(train_m['iou'])
    history['val_iou'   ].append(val_m['iou'])
    history['train_dice'].append(train_m['dice'])
    history['val_dice'  ].append(val_m['dice'])

    print(
        f"Epoch {epoch:02d}/{CONFIG['epochs']}  ({epoch_secs:.0f}s)  |  "
        f"Train  loss={train_loss:.4f}  IoU={train_m['iou']:.4f}  Dice={train_m['dice']:.4f}  |  "
        f"Val    loss={val_loss:.4f}  IoU={val_m['iou']:.4f}  Dice={val_m['dice']:.4f}  "
        f"P={val_m['precision']:.3f}  R={val_m['recall']:.3f}"
    )

    if val_m['iou'] > best_val_iou:
        best_val_iou = val_m['iou']
        best_epoch   = epoch
        best_metrics = {
            'epoch'    : epoch,
            'val_loss' : val_loss,
            **{f'val_{k}': v for k, v in val_m.items()},
        }
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_iou'             : best_val_iou,
            'config'              : CONFIG,
        }, os.path.join(CONFIG['save_dir'], 'best_unet.pth'))
        print(f"  checkpointed  (val IoU = {best_val_iou:.4f})")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  No improvement for {epochs_no_improve}/{patience} epochs")
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

    print()

total_time = datetime.timedelta(seconds=int(time.time() - training_start))
print("=" * 65)
print(f"Training complete  --  total time: {total_time}")
print(f"Best val IoU = {best_val_iou:.4f} at epoch {best_epoch}")
print("=" * 65 + "\n")

torch.save({
    'epoch'           : CONFIG['epochs'],
    'model_state_dict': model.state_dict(),
    'config'          : CONFIG,
}, os.path.join(CONFIG['save_dir'], 'final_unet.pth'))

In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_x, history['train_loss'], label='Train')
axes[0].plot(epochs_x, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs_x, history['train_iou'],  label='Train')
axes[1].plot(epochs_x, history['val_iou'],    label='Val')
axes[1].set_title('IoU'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(epochs_x, history['train_dice'], label='Train')
axes[2].plot(epochs_x, history['val_dice'],   label='Val')
axes[2].set_title('Dice'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.suptitle('UNet Training Curves', fontsize=13)
plt.tight_layout()
curves_path = os.path.join(CONFIG['save_dir'], 'training_curves.png')
plt.savefig(curves_path, dpi=150); plt.close()
print(f"Training curves saved -> {curves_path}")

In [ ]:
print("\nLoading best checkpoint for test evaluation...")
ckpt = torch.load(
    os.path.join(CONFIG['save_dir'], 'best_unet.pth'), map_location=DEVICE
)
model.load_state_dict(ckpt['model_state_dict'])
print(f"  Loaded epoch {ckpt['epoch']}  (val IoU = {ckpt['val_iou']:.4f})")

print("\nRunning test evaluation...")
test_loss, test_m = evaluate(model, test_loader, criterion, DEVICE, desc='test')

print("\n" + "=" * 55)
print("FINAL TEST RESULTS")
print("=" * 55)
print(f"  Loss      : {test_loss:.4f}")
print(f"  IoU       : {test_m['iou']:.4f}")
print(f"  Dice      : {test_m['dice']:.4f}")
print(f"  Precision : {test_m['precision']:.4f}")
print(f"  Recall    : {test_m['recall']:.4f}")
print(f"  F1        : {test_m['f1']:.4f}")
print("=" * 55)

all_results = {
    'best_val': best_metrics,
    'test'    : {'loss': test_loss, **test_m},
    'config'  : CONFIG,
}
metrics_path = os.path.join(CONFIG['save_dir'], 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(all_results, f, indent=4)
print(f"\nMetrics saved -> {metrics_path}")


In [ ]:
def unnormalize(t):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std  = torch.tensor(STD ).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1)

model.eval()
images, masks = next(iter(test_loader))
images, masks = images.to(DEVICE), masks.to(DEVICE)

with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
        logits = model(images)
preds = (torch.sigmoid(logits) > 0.5).float()

n_show = min(4, len(images))
fig, axes = plt.subplots(n_show, 3, figsize=(12, n_show * 4))

for i in range(n_show):
    img_disp  = unnormalize(images[i].cpu()).permute(1, 2, 0).numpy()
    gt_disp   = masks[i].cpu().squeeze().numpy()
    pred_disp = preds[i].cpu().squeeze().numpy()
    iou_val   = compute_metrics(logits[i:i+1].cpu(), masks[i:i+1].cpu())['iou']

    axes[i, 0].imshow(img_disp);                               axes[i, 0].set_title('Input')
    axes[i, 1].imshow(gt_disp,   cmap='gray', vmin=0, vmax=1); axes[i, 1].set_title('Ground Truth')
    axes[i, 2].imshow(pred_disp, cmap='gray', vmin=0, vmax=1); axes[i, 2].set_title(f'Prediction  IoU={iou_val:.3f}')
    for ax in axes[i]: ax.axis('off')

plt.suptitle('UNet Qualitative Results — Test Set', fontsize=13)
plt.tight_layout()
qual_path = os.path.join(CONFIG['save_dir'], 'qualitative_results.png')
plt.savefig(qual_path, dpi=150); plt.close()
print(f"Qualitative results saved -> {qual_path}")

In [ ]:
print("\nRunning threshold sweep on validation set...")
thresholds = np.arange(0.1, 0.91, 0.05)
all_logits_list, all_masks_list = [], []

model.eval()
with torch.no_grad():
    for imgs_b, msks_b in tqdm(val_loader, desc='Collecting val logits', colour='yellow'):
        imgs_b = imgs_b.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            lg = model(imgs_b)
        all_logits_list.append(lg.cpu())
        all_masks_list.append(msks_b)

all_logits_t = torch.cat(all_logits_list)
all_masks_t  = torch.cat(all_masks_list)

sweep_results = []
for t in tqdm(thresholds, desc='Sweeping thresholds', colour='yellow'):
    m = compute_metrics(all_logits_t, all_masks_t, threshold=float(t))
    sweep_results.append((t, m['iou'], m['dice'], m['f1']))

best_t = max(sweep_results, key=lambda x: x[1])
print(f"\nBest threshold : {best_t[0]:.2f}  ->  "
        f"IoU={best_t[1]:.4f}  Dice={best_t[2]:.4f}  F1={best_t[3]:.4f}")

ts, ious, dices, f1s = zip(*sweep_results)
plt.figure(figsize=(8, 4))
plt.plot(ts, ious,  label='IoU')
plt.plot(ts, dices, label='Dice')
plt.plot(ts, f1s,   label='F1')
plt.axvline(best_t[0], color='red', linestyle='--', label=f'Best @ {best_t[0]:.2f}')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.title('Threshold Sweep -- Validation Set')
plt.legend(); plt.tight_layout()
thresh_path = os.path.join(CONFIG['save_dir'], 'threshold_sweep.png')
plt.savefig(thresh_path, dpi=150); plt.close()
print(f"Threshold sweep plot saved -> {thresh_path}")


In [ ]:
def predict_single(image_path: str, threshold: float = 0.5):
    """
    Run inference on one image and save a 3-panel visualisation.
    Usage:
        predict_single('sd2-fr/testing/zebra/110211_mask_random.png_sd2-512_0.png',
                        threshold=best_t[0])
    """
    model.eval()
    img   = Image.open(image_path).convert('RGB')
    img_r = img.resize((CONFIG['image_size'], CONFIG['image_size']), Image.BILINEAR)
    t = TF.to_tensor(img_r)
    t = TF.normalize(t, mean=MEAN, std=STD).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            logit = model(t)
    prob = torch.sigmoid(logit).squeeze().cpu().numpy()
    pred = (prob > threshold).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(img);                               axes[0].set_title('Input')
    axes[1].imshow(prob, cmap='hot',  vmin=0, vmax=1); axes[1].set_title('Probability map')
    axes[2].imshow(pred, cmap='gray', vmin=0, vmax=1); axes[2].set_title(f'Prediction (t={threshold:.2f})')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    out = os.path.join(CONFIG['save_dir'], 'single_inference.png')
    plt.savefig(out, dpi=150); plt.close()
    print(f"Single-image result saved -> {out}")
    return prob

# Uncomment to test on a specific image after training:
# predict_single(
#     'sd2-fr/testing/zebra/110211_mask_random.png_sd2-512_0.png',
#     threshold=best_t[0],
# )

print("\nAll done.")